# Foundation Model Performance Analysis

This notebook performs a comprehensive analysis of various foundation models across multiple medical datasets. It visualizes model performance, creates ranking heatmaps, and compares models based on their parameter counts and performance metrics.

## Contents
1. [Setup and Imports](#setup)
2. [Data Loading](#data-loading)
3. [Data Preparation](#data-preparation)
4. [Performance Visualization](#visualization)
5. [FMCIB Alignment Analysis](#alignment)

<a id='setup'></a>
## 1. Setup and Imports

Import the necessary libraries for data analysis and visualization.

In [96]:
# Standard data manipulation libraries
import pandas as pd
import numpy as np

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set global font size for all plots
font_size = 22

<a id='data-loading'></a>
## 2. Data Loading

Load the performance data from a CSV file and add parameter counts for each model.

In [97]:
# Load the performance data
df = pd.read_csv("/home/suraj/Repositories/TumorImagingBench/notebooks/analysis/overall_results.csv")

# Display the model names to verify data loading
df["Models"]

0              FMCIBExtractor
1               CTFMExtractor
2          CTClipVitExtractor
3              PASTAExtractor
4            VISTA3DExtractor
5               VocoExtractor
6             SUPREMExtractor
7             MerlinExtractor
8    MedImageInsightExtractor
9          ModelsGenExtractor
Name: Models, dtype: object

In [108]:
# Add model parameter counts (in millions)
model_param_counts = {
    "FMCIBExtractor": 184.48,
    "CTFMExtractor": 77.76,
    "CTClipVitExtractor": 25.89,
    "PASTAExtractor": 127.11,
    "VISTA3DExtractor": 174.95,
    "VocoExtractor": 294.86,
    "SUPREMExtractor": 19.07,
    "MerlinExtractor": 270.94,
    "ModelsGenExtractor": 7.03
}

df = df[df["Models"].isin(model_param_counts.keys())]
df["params"] = df["Models"].map(model_param_counts)

In [109]:
df[["Models", "params"]]
df = df.sort_values(by="Models", ascending=True)

<a id='data-preparation'></a>
## 3. Data Preparation

Define functions to prepare and analyze the model performance data.

In [110]:
df

,Unnamed: 0.11,Unnamed: 0.10,Unnamed: 0.9,Unnamed: 0.8,Unnamed: 0.7,Unnamed: 0.6,Unnamed: 0.5,Unnamed: 0.4,Unnamed: 0.3,Unnamed: 0.2,...,FMCIB_alignment_NSCLC_Radiogenomics,VISTA_alignment_NSCLC_Radiogenomics,ModelsGen_alignment_NSCLC_Radiogenomics,FMCIB_alignment_C4KC,VISTA_alignment_C4KC,ModelsGen_alignment_C4KC,FMCIB_alignment_Colorectal_Liver_Metastases,VISTA_alignment_Colorectal_Liver_Metastases,ModelsGen_alignment_Colorectal_Liver_Metastases,params
2,2,2,2,2,2,2,2,2,2,2,...,1.486111,2.187500,2.083333,0.828571,0.928571,0.885714,1.472081,1.898477,1.710660,25.89
1,1,1,1,1,1,1,1,1,1,1,...,2.548611,4.277778,3.784722,1.895238,2.733333,2.376190,2.101523,2.644670,2.335025,77.76
0,0,0,0,0,0,0,0,0,0,0,...,NaN,3.423611,3.423611,NaN,2.795238,2.461905,NaN,3.020305,2.492386,184.48
7,7,7,7,7,7,7,7,7,7,7,...,1.861111,2.006944,1.944444,1.723810,1.876190,1.785714,1.421320,1.791878,1.796954,270.94
9,9,9,9,9,9,9,9,9,9,9,...,3.423611,4.520833,NaN,2.461905,3.776190,NaN,2.492386,4.147208,NaN,7.03
3,3,3,3,3,3,3,3,3,3,3,...,2.201389,3.229167,2.750000,1.742857,2.819048,2.647619,1.802030,3.045685,2.908629,127.11
6,6,6,6,6,6,6,6,6,6,6,...,1.791667,2.493056,2.215278,2.095238,2.647619,2.771429,2.111675,3.375635,3.568528,19.07
4,4,4,4,4,4,4,4,4,4,4,...,3.423611,NaN,4.520833,2.795238,NaN,3.776190,3.020305,NaN,4.147208,174.95
5,5,5,5,5,5,5,5,5,5,5,...,1.020833,0.937500,0.923611,0.476190,0.576190,0.561905,0.654822,0.644670,0.659898,294.86


In [111]:
def prepare_data(df):
    """
    Prepare the dataset for analysis by cleaning, calculating metrics, and adding rankings.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The input dataframe containing model performance data
        
    Returns:
    --------
    pandas.DataFrame
        Processed dataframe with additional metrics and rankings
    """
    # Rename the 'Models' column if needed
    if 'Models' in df.columns:
        df = df.rename(columns={'Models': 'Model'})
    
    # Extract only the model name and dataset columns
    dataset_cols = ['LUNA', 'DLCS', 'NSCLC_Radiomics', 'NSCLC_Radiogenomics', 'C4C_Kits', 'Colorectal_Liver_Metastases']
    necessary_cols = ['Model'] + dataset_cols
    
    # Retain the 'params' column if it exists
    if 'params' in df.columns:
        necessary_cols.append('params')
    
    # Keep only necessary columns (filter out unnamed columns)
    df = df[necessary_cols]
    
    # Calculate average score and standard deviation for each model
    df['Average Score'] = df[dataset_cols].mean(axis=1)
    df['Std Dev'] = df[dataset_cols].std(axis=1)

    df = df.sort_values(by="Model", ascending=False)
    
    # Sort by average score (descending)
    df = df.sort_values('Average Score', ascending=False)
    
    # Reset index for clean numbering
    df = df.reset_index(drop=True)
    
    # Add rank and tier information
    df['Rank'] = range(1, len(df) + 1)
    df['Tier'] = pd.cut(
        df['Average Score'], 
        bins=[0, 0.53, 0.63, 1], 
        labels=['Lower Tier', 'Middle Tier', 'Top Tier'],
        right=False
    )
    
    # Create dataset-specific rank columns
    for col in dataset_cols:
        # Create a new column with the dataset's rank (1 = best)
        df[f"{col}_Rank"] = df[col].rank(ascending=False).astype(int)
    
    return df

<a id='visualization'></a>
## 4. Performance Visualization

Create visualization functions to display model performance as heatmaps and line charts.

In [112]:
def create_rank_heatmap(df):
    """
    Create a heatmap showing the rank of each model across different datasets.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The processed dataframe with rank information
        
    Returns:
    --------
    plotly.graph_objects.Figure
        Heatmap visualization of model ranks
    """
    # Get dataset columns
    df = df.sort_values(by="Model", ascending=True)
    dataset_cols = ['LUNA', 'DLCS', 'NSCLC_Radiomics', 'NSCLC_Radiogenomics', 'C4C_Kits', 'Colorectal_Liver_Metastases']

    rank_cols = [f"{col}_Rank" for col in dataset_cols]
    
    # Create a heatmap using Plotly Express for ranks
    fig = px.imshow(
        df[rank_cols].values,
        labels=dict(x="Dataset", y="Model", color="Rank (1=best)"),
        x=[col.replace('_Rank', '') for col in rank_cols],
        y=df['Model'],
        zmin=1,
        zmax=10,
        color_continuous_scale='Greens_r',  # Reversed scale so darker = better
        aspect="auto",
        title=''
    )
    
    # Add text annotations with the ranks
    for i in range(len(df)):
        for j, col in enumerate(rank_cols):
            rank_value = df[col].iloc[i]
            fig.add_annotation(
                x=j,
                y=i,
                text=f"#{rank_value}",
                showarrow=False,
                font=dict(
                    color='white',
                    size=font_size,
                    weight='normal'
                )
            )
    
    # Update layout with consistent font size
    fig.update_layout(
        height=1000,
        width=1200,
        template='plotly_white',
        font=dict(size=font_size)  # Global font size setting for all text elements
    )
    
    # Hide the colorscale
    fig.update_coloraxes(showscale=False)
    return fig

In [113]:
def create_model_comparison(df):
    """
    Create a line chart comparing model performance across datasets.
    Marker sizes represent model parameter counts.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The processed dataframe with model performance data
        
    Returns:
    --------
    plotly.graph_objects.Figure
        Line chart visualization of model performance
    """
    # Get dataset columns
    dataset_cols = ['LUNA', 'DLCS', 'NSCLC_Radiomics', 'NSCLC_Radiogenomics', 'C4C_Kits', 'Colorectal_Liver_Metastases']
    
    # Create a figure
    fig = go.Figure()
    font_size = 25

    # Determine marker sizes from parameter count if available
    if 'params' in df.columns:
        max_params = df['params'].max()
    else:
        max_params = None
    
    # Define a color palette based on average score
    norm = (df['Average Score'] - df['Average Score'].min()) / (df['Average Score'].max() - df['Average Score'].min())
    greens_palette = px.colors.sequential.Blues
    colors = [greens_palette[int(n * (len(greens_palette) - 1))] for n in norm]
    
    # Add a trace for each model
    for i, row in df.iterrows():
        model_name = row['Model']
        model_data = row[dataset_cols].values
        
        line_style = 'solid'
        width = 2
        
        # Highlight top 3 models with thicker lines
        if row['Rank'] <= 3:
            width = 4
            
        # Add different line styles based on tier
        if row['Tier'] == 'Lower Tier':
            line_style = 'dot'
        elif row['Tier'] == 'Middle Tier':
            line_style = 'dash'
        
        # Determine marker size based on parameters, scaling between 8 and 20
        marker_size = 8
        if max_params is not None:
            marker_size = 8 + (row['params'] / max_params) * 12
        
        # Add trace for this model
        fig.add_trace(go.Scatter(
            x=dataset_cols,
            y=model_data,
            mode='lines+markers',
            name=f"#{row['Rank']} {model_name}",
            line=dict(
                width=width, 
                dash=line_style,
                color=colors[i]
            ),
            marker=dict(size=marker_size),
            hovertemplate="<b>%{fullData.name}</b><br>Dataset: %{x}<br>Score: %{y:.4f}<br>Params: " + str(row['params']) + "<extra></extra>"
        ))
    
    # Add a line for the average performance across models for each dataset
    dataset_avgs = df[dataset_cols].mean()
    fig.add_trace(go.Scatter(
        x=dataset_cols,
        y=dataset_avgs,
        mode='lines+markers',
        name='Average Performance',
        line=dict(width=4, color='black', dash='dot'),
        marker=dict(size=10, color='black', symbol='diamond'),
        hovertemplate="<b>Average Performance</b><br>Dataset: %{x}<br>Score: %{y:.4f}<extra></extra>"
    ))
    
    # Add dummy traces to illustrate marker sizes corresponding to parameter counts if available
    if max_params is not None:
        min_params = df['params'].min()
        median_params = df['params'].median()
        marker_size_low = 8 + (min_params / max_params) * 12
        marker_size_med = 8 + (median_params / max_params) * 12
        marker_size_high = 8 + (max_params / max_params) * 12  # This will be 20
        
        # Dummy trace for Low Params
        fig.add_trace(go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            marker=dict(size=marker_size_low, color="grey"),
            name=f"Low Params ({min_params:.2f})",
            showlegend=True
        ))
        # Dummy trace for Median Params
        fig.add_trace(go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            marker=dict(size=marker_size_med, color="grey"),
            name=f"Median Params ({median_params:.2f})",
            showlegend=True
        ))
        # Dummy trace for High Params
        fig.add_trace(go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            marker=dict(size=marker_size_high, color="grey"),
            name=f"High Params ({max_params:.2f})",
            showlegend=True
        ))
    
    # Update layout
    fig.update_layout(
        title='',
        xaxis_title='Datasets',
        yaxis_title='Performance Score',
        yaxis=dict(range=[0.4, 0.9]),
        legend=dict(
            title='Models (Ranked) & Marker Size Legend',
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='right',
            x=1
        ),
        height=1200,
        width=1200,
        template='plotly_white',
        margin=dict(r=200),  # Extra margin for the legend
        font=dict(size=font_size)
    )
    
    # Add grid lines for better readability
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(0,0,0,0.1)')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(0,0,0,0.1)')
    
    return fig

In [114]:
def main(your_dataframe):
    """
    Main function to run the complete analysis workflow.
    
    Parameters:
    -----------
    your_dataframe : pandas.DataFrame
        The input dataframe containing model performance data
        
    Returns:
    --------
    pandas.DataFrame
        Processed dataframe with additional metrics and rankings
    """
    # Prepare the data
    clean_df = prepare_data(your_dataframe)
    
    # Display the processed dataframe
    print("\nProcessed Performance Data:")
    display(clean_df)
    
    # Create the plots
    print("\nGenerating visualizations...")
    rank_heatmap = create_rank_heatmap(clean_df)
    model_comparison = create_model_comparison(clean_df)
    
    # Show the plots
    print("\nRank Heatmap - Shows each model's ranking across datasets:")
    rank_heatmap.show()
    
    print("\nModel Comparison - Shows detailed performance with parameter information:")
    model_comparison.show()
    
    # Save the plots
    print("\nSaving visualizations...")
    rank_heatmap.write_html("model_performance_rank_heatmap.html")
    model_comparison.write_html("model_performance_by_model.html")
    print("Visualizations saved as HTML files.")
    
    return clean_df  # Return the processed dataframe for further use if needed

### Run the Main Analysis

Execute the main analysis function to process the data and generate visualizations.

In [115]:
# Run the main analysis
processed_df = main(df)


Processed Performance Data:


,Model,LUNA,DLCS,NSCLC_Radiomics,NSCLC_Radiogenomics,C4C_Kits,Colorectal_Liver_Metastases,params,Average Score,Std Dev,Rank,Tier,LUNA_Rank,DLCS_Rank,NSCLC_Radiomics_Rank,NSCLC_Radiogenomics_Rank,C4C_Kits_Rank,Colorectal_Liver_Metastases_Rank
0,FMCIBExtractor,0.886008,0.675796,0.577038,0.587946,0.686944,0.577222,184.48,0.665159,0.119014,1,Top Tier,1,1,3,5,3,1
1,ModelsGenExtractor,0.806061,0.645320,0.577361,0.609598,0.733611,0.530185,7.03,0.650356,0.102598,2,Top Tier,2,2,2,4,1,2
2,VISTA3DExtractor,0.711121,0.607937,0.582637,0.622098,0.681667,0.487778,174.95,0.615540,0.078759,3,Middle Tier,3,3,1,1,4,4
3,SUPREMExtractor,0.645022,0.544219,0.560617,0.556027,0.718611,0.482037,19.07,0.584422,0.083846,4,Middle Tier,6,7,5,7,2,5
4,MerlinExtractor,0.637643,0.561696,0.569879,0.612946,0.642500,0.431296,270.94,0.575994,0.078445,5,Middle Tier,7,5,4,3,5,8
5,PASTAExtractor,0.664839,0.556302,0.557108,0.569866,0.604167,0.464815,127.11,0.569516,0.065693,6,Middle Tier,4,6,6,6,6,6
6,CTFMExtractor,0.654310,0.592034,0.544208,0.620424,0.463333,0.452778,77.76,0.554515,0.083044,7,Middle Tier,5,4,7,2,9,7
7,CTClipVitExtractor,0.572564,0.494423,0.449703,0.510379,0.492500,0.495741,25.89,0.502552,0.039923,8,Lower Tier,8,9,9,8,8,3
8,VocoExtractor,0.493761,0.507139,0.526161,0.461384,0.563333,0.420741,294.86,0.495420,0.049865,9,Lower Tier,9,8,8,9,7,9



Generating visualizations...

Rank Heatmap - Shows each model's ranking across datasets:



Model Comparison - Shows detailed performance with parameter information:



Saving visualizations...
Visualizations saved as HTML files.


<a id='alignment'></a>
## 5. FMCIB Alignment Analysis

Analyze the alignment between different models and the FMCIB (Foundation Model for Cancer Image Biomarkers) model.

In [127]:
# Exclude the row corresponding to FMCIBExtractor
df_alignment = df[df["Models"] != "ModelsGenExtractor"].copy()

# Compute the average alignment across the specified datasets for each model
df_alignment["FMCIB_alignment_avg"] = df_alignment[df.filter(like="ModelsGen_alignment").columns].mean(axis=1)

# Display the alignment data
print("\nFMCIB Alignment Data:")
display(df_alignment[["Models", "FMCIB_alignment_avg"] + list(df.filter(like="ModelsGen_alignment").columns)])


FMCIB Alignment Data:


,Models,FMCIB_alignment_avg,ModelsGen_alignment_LUNA,ModelsGen_alignment_DLCS,ModelsGen_alignment_NSCLC_Radiomics,ModelsGen_alignment_NSCLC_Radiogenomics,ModelsGen_alignment_C4KC,ModelsGen_alignment_Colorectal_Liver_Metastases
2,CTClipVitExtractor,1.411936,1.586411,1.264877,0.940618,2.083333,0.885714,1.710660
1,CTFMExtractor,2.786830,3.265879,2.992415,1.966746,3.784722,2.376190,2.335025
0,FMCIBExtractor,2.690975,2.707533,2.775379,2.285036,3.423611,2.461905,2.492386
7,MerlinExtractor,1.437609,1.091581,0.976079,1.030879,1.944444,1.785714,1.796954
3,PASTAExtractor,2.425147,2.415066,2.100350,1.729216,2.750000,2.647619,2.908629
6,SUPREMExtractor,2.500538,2.416544,2.668028,1.363420,2.215278,2.771429,3.568528
4,VISTA3DExtractor,3.890591,4.197932,3.995916,2.705463,4.520833,3.776190,4.147208
5,VocoExtractor,0.463055,0.205318,0.097433,0.330166,0.923611,0.561905,0.659898


In [128]:
df_alignment

,Unnamed: 0.11,Unnamed: 0.10,Unnamed: 0.9,Unnamed: 0.8,Unnamed: 0.7,Unnamed: 0.6,Unnamed: 0.5,Unnamed: 0.4,Unnamed: 0.3,Unnamed: 0.2,...,VISTA_alignment_NSCLC_Radiogenomics,ModelsGen_alignment_NSCLC_Radiogenomics,FMCIB_alignment_C4KC,VISTA_alignment_C4KC,ModelsGen_alignment_C4KC,FMCIB_alignment_Colorectal_Liver_Metastases,VISTA_alignment_Colorectal_Liver_Metastases,ModelsGen_alignment_Colorectal_Liver_Metastases,params,FMCIB_alignment_avg
2,2,2,2,2,2,2,2,2,2,2,...,2.187500,2.083333,0.828571,0.928571,0.885714,1.472081,1.898477,1.710660,25.89,1.411936
1,1,1,1,1,1,1,1,1,1,1,...,4.277778,3.784722,1.895238,2.733333,2.376190,2.101523,2.644670,2.335025,77.76,2.786830
0,0,0,0,0,0,0,0,0,0,0,...,3.423611,3.423611,NaN,2.795238,2.461905,NaN,3.020305,2.492386,184.48,2.690975
7,7,7,7,7,7,7,7,7,7,7,...,2.006944,1.944444,1.723810,1.876190,1.785714,1.421320,1.791878,1.796954,270.94,1.437609
3,3,3,3,3,3,3,3,3,3,3,...,3.229167,2.750000,1.742857,2.819048,2.647619,1.802030,3.045685,2.908629,127.11,2.425147
6,6,6,6,6,6,6,6,6,6,6,...,2.493056,2.215278,2.095238,2.647619,2.771429,2.111675,3.375635,3.568528,19.07,2.500538
4,4,4,4,4,4,4,4,4,4,4,...,NaN,4.520833,2.795238,NaN,3.776190,3.020305,NaN,4.147208,174.95,3.890591
5,5,5,5,5,5,5,5,5,5,5,...,0.937500,0.923611,0.476190,0.576190,0.561905,0.654822,0.644670,0.659898,294.86,0.463055


### Visualize FMCIB Alignment

Create a bar plot to visualize the average FMCIB alignment for different models.

In [130]:
# Create a minimalist bar plot of averaged FMCIB alignment for different models
fig = px.bar(
    df_alignment,
    x="Models",
    y="FMCIB_alignment_avg",
    labels={"FMCIB_alignment_avg": "Average FMCIB Alignment", "Models": "Model"},
    title="FMCIB Alignment Comparison",
)

# Style the plot
fig.update_traces(
    marker_color="#FF7C5B",
    marker_opacity=0.8,
)

# Update the layout to a minimalist style
fig.update_layout(
    template="simple_white",
    title_x=0.5,
    xaxis_title="Model",
    yaxis_title="Average Alignment",
    width=1200,
    height=1200,
    font_size=30,
    showlegend=False
)

# Display and save the plot
fig.show()
fig.write_html("fmcib_alignment_comparison.html")
print("FMCIB alignment visualization saved as HTML file.")

FMCIB alignment visualization saved as HTML file.


## Conclusion

This notebook provides a comprehensive analysis of model performance across multiple datasets, with visualizations to help identify patterns and trends. The key takeaways include:

1. **Rankings**: The heatmap shows how each model ranks across different datasets, helping to identify the most consistently performant models.

2. **Performance vs. Size**: The line chart illustrates the relationship between model size (parameter count) and performance, helping to identify efficient models.

3. **FMCIB Alignment**: The bar chart shows how different models align with the FMCIB model, providing insights into model similarity and potential knowledge transfer.

These visualizations can guide decisions about model selection for specific medical imaging tasks, considering both performance and computational requirements.